In [0]:
df = spark.read.csv("/databricks-datasets/adult/adult.data", header=False, inferSchema=True)
df.show(5)

+---+-----------------+--------+----------+----+-------------------+------------------+--------------+------+-------+------+----+----+--------------+------+
|_c0|              _c1|     _c2|       _c3| _c4|                _c5|               _c6|           _c7|   _c8|    _c9|  _c10|_c11|_c12|          _c13|  _c14|
+---+-----------------+--------+----------+----+-------------------+------------------+--------------+------+-------+------+----+----+--------------+------+
| 39|        State-gov| 77516.0| Bachelors|13.0|      Never-married|      Adm-clerical| Not-in-family| White|   Male|2174.0| 0.0|40.0| United-States| <=50K|
| 50| Self-emp-not-inc| 83311.0| Bachelors|13.0| Married-civ-spouse|   Exec-managerial|       Husband| White|   Male|   0.0| 0.0|13.0| United-States| <=50K|
| 38|          Private|215646.0|   HS-grad| 9.0|           Divorced| Handlers-cleaners| Not-in-family| White|   Male|   0.0| 0.0|40.0| United-States| <=50K|
| 53|          Private|234721.0|      11th| 7.0| Married-c

## Step 1: Load and inspect a sample dataset

Loading the Adult Census Income dataset (bundled with Databricks Community/Free Edition sample datasets) to demonstrate reading and inspecting data with PySpark. Columns are auto-named since the raw file has no header row.

In [0]:
df.printSchema()


root
 |-- _c0: integer (nullable = true)
 |-- _c1: string (nullable = true)
 |-- _c2: double (nullable = true)
 |-- _c3: string (nullable = true)
 |-- _c4: double (nullable = true)
 |-- _c5: string (nullable = true)
 |-- _c6: string (nullable = true)
 |-- _c7: string (nullable = true)
 |-- _c8: string (nullable = true)
 |-- _c9: string (nullable = true)
 |-- _c10: double (nullable = true)
 |-- _c11: double (nullable = true)
 |-- _c12: double (nullable = true)
 |-- _c13: string (nullable = true)
 |-- _c14: string (nullable = true)



## Step 2: Rename columns for clarity

In [0]:
columns = ["age", "workclass", "fnlwgt", "education", "education_num",
           "marital_status", "occupation", "relationship", "race", "sex",
           "capital_gain", "capital_loss", "hours_per_week", "native_country", "income"]
df = df.toDF(*columns)
df.select("age", "workclass", "education", "occupation", "hours_per_week", "income").show(5)

+---+-----------------+----------+------------------+--------------+------+
|age|        workclass| education|        occupation|hours_per_week|income|
+---+-----------------+----------+------------------+--------------+------+
| 39|        State-gov| Bachelors|      Adm-clerical|          40.0| <=50K|
| 50| Self-emp-not-inc| Bachelors|   Exec-managerial|          13.0| <=50K|
| 38|          Private|   HS-grad| Handlers-cleaners|          40.0| <=50K|
| 53|          Private|      11th| Handlers-cleaners|          40.0| <=50K|
| 28|          Private| Bachelors|    Prof-specialty|          40.0| <=50K|
+---+-----------------+----------+------------------+--------------+------+
only showing top 5 rows


## Step 3: Basic PySpark operations — filtering and aggregation

Demonstrating core DataFrame operations: filtering rows by a condition, and aggregating data with groupBy — the everyday building blocks of exploring a dataset in Spark.

In [0]:
# Filter: workers over 40 hours/week
overtime = df.filter(df.hours_per_week > 40)
print(f"Workers logging more than 40 hours/week: {overtime.count()} out of {df.count()}")
overtime.select("age", "occupation", "hours_per_week", "income").show(5)

Workers logging more than 40 hours/week: 9581 out of 32561
+---+-----------------+--------------+------+
|age|       occupation|hours_per_week|income|
+---+-----------------+--------------+------+
| 52|  Exec-managerial|          45.0|  >50K|
| 31|   Prof-specialty|          50.0|  >50K|
| 37|  Exec-managerial|          80.0|  >50K|
| 32|            Sales|          50.0| <=50K|
| 34| Transport-moving|          45.0| <=50K|
+---+-----------------+--------------+------+
only showing top 5 rows


In [0]:
# Aggregation: average hours worked per education level
df.groupBy("education").avg("hours_per_week").orderBy("avg(hours_per_week)", ascending=False).show(10)

+-------------+-------------------+
|    education|avg(hours_per_week)|
+-------------+-------------------+
|  Prof-school|  47.42534722222222|
|    Doctorate| 46.973365617433416|
|      Masters|  43.83633197910621|
|    Bachelors| 42.614005602240894|
|    Assoc-voc|  41.61070911722142|
|      HS-grad| 40.575373773926295|
|   Assoc-acdm| 40.504217432052485|
|      7th-8th|  39.36687306501548|
|      5th-6th|   38.8978978978979|
| Some-college|  38.85228363736113|
+-------------+-------------------+
only showing top 10 rows


## Step 4: A simple ML step — Linear Regression with MLlib

Building a basic linear regression model to predict hours worked per week from age and education level, demonstrating the ability to run an ML pipeline inside Databricks' distributed environment. This is intentionally simple — the goal is showing the workflow, not building a sophisticated model.

In [0]:
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import LinearRegression

assembler = VectorAssembler(inputCols=["age", "education_num"], outputCol="features")
data = assembler.transform(df).select("features", "hours_per_week")

train_data, test_data = data.randomSplit([0.8, 0.2], seed=42)

lr = LinearRegression(featuresCol="features", labelCol="hours_per_week")
model = lr.fit(train_data)

print(f"Coefficients: {model.coefficients}")
print(f"Intercept: {model.intercept}")

summary = model.evaluate(test_data)
print(f"RMSE on test data: {summary.rootMeanSquaredError:.2f}")
print(f"R2 on test data: {summary.r2:.4f}")

Coefficients: [0.05549409388241152,0.6932756884613364]
Intercept: 31.316600838234027
RMSE on test data: 12.26
R2 on test data: 0.0285


### Interpreting the result

An R² of ~0.03 means age and education level alone explain very little of the variance in hours worked per week — not surprising, since hours worked is driven by many other factors (job type, contract, personal circumstances) not captured by these two features. This is a useful, honest result: it demonstrates running a full ML pipeline (feature assembly, train/test split, model fit, evaluation) correctly, even though the model itself has low predictive power with this simple feature set.

## Summary

This notebook demonstrates hands-on use of the Databricks notebook environment: loading and inspecting a sample dataset with PySpark, renaming columns, filtering and aggregating data, and running a basic MLlib linear regression with proper train/test evaluation. Built as personal, hands-on exposure to the Databricks/PySpark workflow — not production-scale experience.